# AF2WIN — isolated Kaggle arm
One arm only; validates exact dataset/checkpoint SHA, writes `last.pt` every epoch, and never accesses test.

In [ ]:
import csv, os, shutil, subprocess, sys, time
from pathlib import Path
WORK=Path('/kaggle/working'); REPO=WORK/'coffee-bean-detection'; INPUT=Path('/kaggle/input'); ARM='AF2WIN'; SEED=42
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    result=subprocess.run(['git','clone','--depth','1','--branch','agent/af2-spectral-factorization','https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True); os.chdir(REPO)
sys.path.insert(0,str(REPO/'src'))
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'): sys.modules.pop(module_name,None)
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input, restore_spectral_kaggle_run
from coffee_detector.af2_spectral.audit import run_spectral_static_audit
DATA,ARTIFACTS,_=prepare_af2_spectral_kaggle_input(INPUT,WORK); OUT=WORK/'af2-spectral-factorization-v1'; OUT.mkdir(exist_ok=True)
STATIC=OUT/'static_audit.json'; audit=run_spectral_static_audit(ARTIFACTS['D0_seed42_best.pt'],STATIC,device='cuda:0'); assert audit['decision']=='PASS'
CONFIG=REPO/'configs/af2_spectral/AF2WIN_yolo26n.yaml'
restored=restore_spectral_kaggle_run(INPUT,OUT,arm=ARM,seed=SEED,d0_checkpoint=ARTIFACTS['D0_seed42_best.pt'],config=CONFIG); print('RESTORED:',restored)

In [ ]:
LOG=OUT/f'{ARM}_seed{SEED}_run.log'; RESULT=OUT/'val_reports'/f'{ARM}_seed{SEED}_result.json'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_spectral_arm','--arm',ARM,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(ARTIFACTS['D0_seed42_best.pt']),'--static-audit',str(STATIC),'--output-root',str(OUT),'--seed',str(SEED),'--device','0','--authorize-training']
if not RESULT.is_file():
    with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    reported=None
    while process.poll() is None:
        csv_path=OUT/ARM/f'{ARM}_seed{SEED}'/'results.csv'
        epoch=sum(1 for _ in csv_path.open())-1 if csv_path.is_file() else 0
        if epoch!=reported: print(f'{ARM}: {epoch}/50 epoch | log={LOG}',flush=True); reported=epoch
        time.sleep(120)
    if process.returncode: print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
assert RESULT.is_file(), RESULT
print(RESULT.read_text())
archive=shutil.make_archive(f'/kaggle/working/{ARM}_seed{SEED}_output','zip',OUT); print('DOWNLOAD SEBELUM STOP SESSION:',archive)